In [ ]:
# ============================================================
# 1- Database design in phpMyAdmin -> SQL codes
# ============================================================

In [1]:
# ============================================================
# 2- Phone Number Normalization
# ============================================================

import re

_PERSIAN_DIGITS = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789"
)

_ARABIC_DIGITS = str.maketrans(
    "٠١٢٣٤٥٦٧٨٩",
    "0123456789"
)


def normalize_iran_phone(value) -> str:
    """
    Return an Iranian mobile number in local 10-digit form.

    Example:
        9121234567

    Accepted formats:
        9121234567
        09121234567
        +989121234567
        00989121234567

    Persian and Arabic-Indic digits are also accepted.
    """

    if value is None:
        raise ValueError("Phone number is empty")

    # Normalize Unicode digits first.
    s = str(value).strip()
    s = s.translate(_PERSIAN_DIGITS)
    s = s.translate(_ARABIC_DIGITS)

    # Reject obvious Excel numeric artifacts.
    if re.fullmatch(r"\d+\.0+", s):
        raise ValueError(
            f"Phone number appears to have been converted to a numeric "
            f"Excel value: {value!r}"
        )

    # Keep digits and an optional leading '+'.
    s = re.sub(r"[^\d+]", "", s)

    # Convert international Iranian formats to local format.
    if s.startswith("+98"):
        s = s[3:]
    elif s.startswith("0098"):
        s = s[4:]
    elif s.startswith("98") and len(s) == 12:
        s = s[2:]
    elif s.startswith("0"):
        s = s[1:]

    # Final Iranian mobile validation.
    if not re.fullmatch(r"9\d{9}", s):
        raise ValueError(
            f"Invalid Iranian mobile number: {value!r}"
        )

    return s

In [2]:
# ============================================================
# 3- Import Excel Data into the Database
# ============================================================
from pathlib import Path

import pandas as pd
import mysql.connector

EXCEL_FILE = Path(
    "/home/saeed/repositories/contacts/import_eitaa/contacts-1-44.xlsx"
)


def get_db_connection():
    return mysql.connector.connect(
        host="127.0.0.1",
        port=3306,
        user="reza",
        password="123",
        database="eitaa_import",
        charset="utf8mb4",
    )


def import_excel():
    df = pd.read_excel(
        EXCEL_FILE,
        dtype={
            "Full Name": "string",
            "Phone Number": "string",
        },
    )

    expected = {"Full Name", "Phone Number"}

    if set(df.columns) != expected:
        raise ValueError(
            f"Expected columns {expected}, got {set(df.columns)}"
        )

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        sql = """
            INSERT INTO contacts (
                full_name,
                phone_original,
                phone_normalized,
                status
            )
            VALUES (%s, %s, %s, 'pending')
            ON DUPLICATE KEY UPDATE
                full_name = VALUES(full_name),
                phone_original = VALUES(phone_original)
        """

        processed = 0
        invalid_phones = 0
        empty_names = 0

        for _, row in df.iterrows():

            # ------------------------------------------------
            # Full name
            # ------------------------------------------------
            if pd.isna(row["Full Name"]):
                empty_names += 1
                continue

            full_name = str(row["Full Name"]).strip()

            if not full_name:
                empty_names += 1
                continue

            # ------------------------------------------------
            # Phone number
            # ------------------------------------------------
            if pd.isna(row["Phone Number"]):
                invalid_phones += 1
                print("Invalid phone number: empty Excel cell")
                continue

            phone_original = str(row["Phone Number"]).strip()

            try:
                phone_normalized = normalize_iran_phone(
                    phone_original
                )
            except ValueError as exc:
                invalid_phones += 1
                print(
                    f"Invalid phone number "
                    f"{phone_original!r}: {exc}"
                )
                continue

            # ------------------------------------------------
            # Insert/update database
            # ------------------------------------------------
            cursor.execute(
                sql,
                (
                    full_name,
                    phone_original,
                    phone_normalized,
                ),
            )

            processed += 1

        conn.commit()

        # ----------------------------------------------------
        # Verify database state after import
        # ----------------------------------------------------
        cursor.execute(
            """
            SELECT
                COUNT(*) AS total,
                SUM(status = 'pending') AS pending,
                SUM(status = 'processing') AS processing,
                SUM(status = 'success') AS success,
                SUM(status = 'skipped_existing') AS skipped_existing,
                SUM(status = 'failed') AS failed
            FROM contacts
            """
        )

        stats = cursor.fetchone()

        print()
        print("========== IMPORT RESULT ==========")
        print(f"Excel rows processed : {processed}")
        print(f"Invalid phones       : {invalid_phones}")
        print(f"Empty names          : {empty_names}")
        print()
        print(f"DB total             : {stats[0]}")
        print(f"DB pending           : {stats[1]}")
        print(f"DB processing        : {stats[2]}")
        print(f"DB success           : {stats[3]}")
        print(f"DB skipped_existing  : {stats[4]}")
        print(f"DB failed            : {stats[5]}")
        print("====================================")

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


if __name__ == "__main__":
    import_excel()



========== IMPORT RESULT ==========
Excel rows processed : 44
Invalid phones       : 0
Empty names          : 0

DB total             : 44
DB pending           : 44
DB processing        : 0
DB success           : 0
DB skipped_existing  : 0
DB failed            : 0


In [3]:
# ============================================================
# 4- Chrome session persistence
# ============================================================
# Clean, isolated Selenium Chrome session

from pathlib import Path
from tempfile import mkdtemp
from typing import Optional
import shutil

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.webdriver import WebDriver

# ChromeDriver

CHROMEDRIVER = (
        Path.cwd()
        / "drivers"
        / "chromedriver"
        / "chromedriver"
)

# Create a completely new temporary Chrome User Data Directory

TEMP_CHROME_PROFILE = Path(
    mkdtemp(prefix="selenium-chrome-")
)

# Close previous Selenium session, if one exists

previous_driver: Optional[WebDriver] = globals().get("driver")

if previous_driver is not None:
    try:
        previous_driver.quit()
        print("\nPrevious Selenium driver closed.")
    except Exception as e:
        print("\nPrevious Selenium driver could not be closed:")
        print(repr(e))

# Chrome options

options = Options()

options.add_argument(
    f"--user-data-dir={TEMP_CHROME_PROFILE}"
)

options.add_argument("--start-maximized")

#  Explicit manual ChromeDriver
#  Selenium Manager is not used for driver discovery.


service = Service(
    executable_path=str(CHROMEDRIVER)
)

# Create new isolated Chrome session


driver: WebDriver = webdriver.Chrome(
    service=service,
    options=options,
)

driver.set_page_load_timeout(60)


First run:

driver = create_driver()

driver.get("https://web.eitaa.com/")

Log into Eitaa manually.

Close the browser normally.

On subsequent executions Selenium will reuse that Chrome profile/session.

This is much more robust than trying to automate OTP authentication.

In [5]:
# ============================================================
# 5- Close Selenium and remove temporary Chrome profile
# ============================================================

if "driver" in globals() and driver is not None:
    try:
        driver.quit()
        print("Selenium Chrome session closed.")
    finally:
        driver = None

if "TEMP_CHROME_PROFILE" in globals():
    shutil.rmtree(
        TEMP_CHROME_PROFILE,
        ignore_errors=True
    )

    print(
        "Temporary Selenium Chrome profile removed."
    )

Selenium Chrome session closed.
Temporary Selenium Chrome profile removed.


In [4]:
# ============================================================
# 6- Selenium Selectors
# ============================================================
# To test that the selector is unique:
# allow pasting
# document.querySelectorAll(".btn-menu-item.tgico-user.rp").length

from selenium.webdriver.common.by import By


class Selectors:
    # 1. Hamburger menu
    HAMBURGER_MENU = (
        By.CSS_SELECTOR,
        ".btn-icon.btn-menu-toggle.rp.sidebar-tools-button.is-visible"
    )

    # 2. Contacts menu item
    CONTACTS_BUTTON = (
        By.XPATH,
        "//div[contains(@class, 'btn-menu-item') and .//span[normalize-space()='Contacts']]"
    )

    # 3. "+" button inside Contacts
    ADD_CONTACT_BUTTON = (
        By.CSS_SELECTOR,
        "button.tgico-add.is-visible"
    )

    # 4. First name field
    FIRST_NAME = (
        By.CSS_SELECTOR,
        'div.input-field:first-child > div.input-field-input[contenteditable="true"]'
    )

    # 5. Phone number field
    PHONE = (
        By.CSS_SELECTOR,
        'div.input-field-phone > '
        'div.input-field-input[contenteditable="true"]'
    )

    # 6. Add/Save button
    SAVE_BUTTON = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-primary.btn-color-primary"
    )

    # 7. Close Add Contact window ( × button )
    CLOSE_CONTACT_WINDOW = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-icon.popup-close"
    )

In [5]:
# ============================================================
# 7- Reliable waits
# ============================================================
#
# Use explicit waits instead of time.sleep().
#
# The purpose of this section is to provide reusable,
# reliable synchronization helpers for the Selenium code
# used in later sections.
#
# These helpers are designed for situations such as:
#   - waiting for Chrome/Eitaa elements to appear
#   - waiting for an element to become visible
#   - waiting for an element to become clickable
#   - waiting for an element/popup to disappear
#   - waiting for a specific Selenium condition
#   - handling temporary/stale DOM elements
#
# ============================================================

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    StaleElementReferenceException,
    NoSuchElementException,
    ElementClickInterceptedException,
)


# ------------------------------------------------------------
# General configuration
# ------------------------------------------------------------

WAIT_SECONDS = 30


# ------------------------------------------------------------
# 1. Create a WebDriverWait object
# ------------------------------------------------------------
#
# Keeping this in one function makes it possible to change
# the global timeout later without changing all wait helpers.

def get_wait(driver, timeout=WAIT_SECONDS):
    """
    Return a WebDriverWait instance.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    timeout : int | float
        Maximum number of seconds to wait.

    Returns
    -------
    WebDriverWait
    """
    return WebDriverWait(
        driver,
        timeout,
        poll_frequency=0.2,
        ignored_exceptions=(
            NoSuchElementException,
            StaleElementReferenceException,
        ),
    )


# ------------------------------------------------------------
# 2. Wait until an element exists in the DOM
# ------------------------------------------------------------

def wait_present(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is present in the DOM.

    The element does not necessarily have to be visible.

    Useful when:
        - the element exists but is not yet displayed
        - another wait will handle visibility/clickability
    """

    return get_wait(driver, timeout).until(
        EC.presence_of_element_located(locator)
    )


# ------------------------------------------------------------
# 3. Wait until an element is visible
# ------------------------------------------------------------

def wait_visible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is present and visible.

    Returns
    -------
    WebElement
    """

    return get_wait(driver, timeout).until(
        EC.visibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 4. Wait until an element is clickable
# ------------------------------------------------------------

def wait_clickable(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is visible and enabled/clickable.

    This should normally be used before clicking buttons,
    links, menu items, etc.
    """

    return get_wait(driver, timeout).until(
        EC.element_to_be_clickable(locator)
    )


# ------------------------------------------------------------
# 5. Wait until an element disappears
# ------------------------------------------------------------

def wait_invisible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is no longer visible.

    This is particularly useful for:
        - closing popups
        - waiting for dialogs to disappear
        - waiting for loading overlays to disappear

    Note:
        Selenium's invisibility_of_element_located() also
        succeeds if the element is removed from the DOM.
    """

    return get_wait(driver, timeout).until(
        EC.invisibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 6. Wait until an element disappears from the DOM
# ------------------------------------------------------------

def wait_stale(driver, element, timeout=WAIT_SECONDS):
    """
    Wait until a previously obtained WebElement becomes stale.

    Useful when an existing element is replaced/removed after
    an operation such as saving a form.
    """

    return get_wait(driver, timeout).until(
        EC.staleness_of(element)
    )


# ------------------------------------------------------------
# 7. Wait for a locator to contain specific text
# ------------------------------------------------------------

def wait_text(driver, locator, text, timeout=WAIT_SECONDS):
    """
    Wait until an element contains the specified text.
    """

    return get_wait(driver, timeout).until(
        EC.text_to_be_present_in_element(
            locator,
            text,
        )
    )


# ------------------------------------------------------------
# 8. Wait for a specific attribute value
# ------------------------------------------------------------

def wait_attribute(
    driver,
    locator,
    attribute,
    value,
    timeout=WAIT_SECONDS,
):
    """
    Wait until an element has the specified attribute value.

    Example:
        wait_attribute(
            driver,
            Selectors.SOME_ELEMENT,
            "class",
            "active",
        )
    """

    return get_wait(driver, timeout).until(
        EC.attribute_to_include(
            locator,
            value,
        )
    )


# ------------------------------------------------------------
# 9. Wait until an element is selected
# ------------------------------------------------------------

def wait_selected(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until a selectable element is selected.
    """

    return get_wait(driver, timeout).until(
        EC.element_located_to_be_selected(locator)
    )


# ------------------------------------------------------------
# 10. Wait for a custom condition
# ------------------------------------------------------------

def wait_until(
    driver,
    condition,
    timeout=WAIT_SECONDS,
):
    """
    Wait until a custom Selenium condition returns a truthy
    value.

    The condition receives the driver.

    Example:

        wait_until(
            driver,
            lambda d: some_condition(d)
        )

    This is useful when Selenium's built-in expected
    conditions are not sufficient.
    """

    return get_wait(driver, timeout).until(condition)


# ------------------------------------------------------------
# 11. Safe click helper
# ------------------------------------------------------------

def wait_and_click(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is clickable and then click it.

    Returns
    -------
    WebElement
        The element that was clicked.
    """

    element = wait_clickable(
        driver,
        locator,
        timeout,
    )

    element.click()

    return element


# ------------------------------------------------------------
# 12. Wait for a popup to close
# ------------------------------------------------------------

def wait_popup_closed(
    driver,
    popup_locator,
    timeout=WAIT_SECONDS,
):
    """
    Wait until a popup/dialog is no longer visible.

    This is a convenience wrapper around
    wait_invisible().

    It is intended for situations such as the Eitaa
    Add Contact popup.
    """

    return wait_invisible(
        driver,
        popup_locator,
        timeout,
    )


# ------------------------------------------------------------
# 13. Wait for an element and then enter text
# ------------------------------------------------------------

def wait_and_type(
    driver,
    locator,
    text,
    timeout=WAIT_SECONDS,
    clear_first=True,
):
    """
    Wait for an input field to become visible and type text.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium driver.

    locator : tuple
        Selenium locator, for example:
            (By.CSS_SELECTOR, "...")

    text : str
        Text to enter.

    clear_first : bool
        If True, clear the input before typing.

    Returns
    -------
    WebElement
        The input element.
    """

    element = wait_visible(
        driver,
        locator,
        timeout,
    )

    if clear_first:
        element.clear()

    element.send_keys(text)

    return element


In [6]:
# ============================================================
# 8- Add contact to Eitaa
# ============================================================
#
# This section adds ONE contact to Eitaa and determines the
# result:
#
#     "success"  -> contact was added
#     "skipped"  -> phone number is not registered on Eitaa
#     "failed"   -> an unexpected error occurred
#
# Section 8 does NOT update the database.
#
# Database updates and retry handling belong to Sections 9
# and 10.
#
# ============================================================

from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException,
)


# ------------------------------------------------------------
# Result constants
# ------------------------------------------------------------

CONTACT_SUCCESS = "success"
CONTACT_SKIPPED = "skipped"
CONTACT_FAILED = "failed"


# ------------------------------------------------------------
# Eitaa messages
# ------------------------------------------------------------

UNREGISTERED_PHONE_TEXT = (
    "The person with this phone number is not registered on Eitaa yet."
)


# ------------------------------------------------------------
# Add Contact popup
# ------------------------------------------------------------
#
# This is the popup locator already used successfully in the
# tested code supplied for this project.
#
# It is deliberately kept here instead of adding a new
# untested selector to the Selectors class.

ADD_CONTACT_POPUP = (
    By.CSS_SELECTOR,
    ".popup-create-contact.active",
)


# ------------------------------------------------------------
# Helper: check whether Contacts page is active
# ------------------------------------------------------------

def contacts_page_is_active(driver):
    """
    Return True if the Contacts container currently has
    the 'active' class.
    """

    contacts_containers = driver.find_elements(
        By.CSS_SELECTOR,
        "#contacts-container",
    )

    if not contacts_containers:
        return False

    contacts_container = contacts_containers[0]

    try:
        classes = contacts_container.get_attribute("class") or ""
    except WebDriverException:
        return False

    return "active" in classes.split()


# ------------------------------------------------------------
# Helper: open Contacts page
# ------------------------------------------------------------

def ensure_contacts_page(driver):
    """
    Make sure Eitaa is currently showing the Contacts page.

    If Contacts is already active, nothing is clicked.

    Otherwise:
        1. Open the hamburger menu.
        2. Click Contacts.
        3. Wait until Contacts becomes active.
    """

    if contacts_page_is_active(driver):
        return

    wait_and_click(
        driver,
        Selectors.HAMBURGER_MENU,
    )

    wait_and_click(
        driver,
        Selectors.CONTACTS_BUTTON,
    )

    wait_until(
        driver,
        lambda d: contacts_page_is_active(d),
    )


# ------------------------------------------------------------
# Helper: detect unregistered-phone message
# ------------------------------------------------------------

def unregistered_phone_message_is_visible(driver):
    """
    Return True when the Eitaa 'not registered' message is
    visible.

    The exact popup selector for this message was not provided
    in Section 6, so this condition searches visible page
    elements for the exact message text documented in the
    project specification.
    """

    elements = driver.find_elements(
        By.XPATH,
        "//*[contains(normalize-space(.), "
        f"{UNREGISTERED_PHONE_TEXT!r}"
        ")]",
    )

    for element in elements:
        try:
            if element.is_displayed():
                return True
        except WebDriverException:
            continue

    return False


# ------------------------------------------------------------
# Helper: wait for the result of Save
# ------------------------------------------------------------

def wait_for_add_contact_result(driver, timeout=WAIT_SECONDS):
    """
    Wait for Eitaa to produce the result of the Add Contact
    operation.

    Returns:
        CONTACT_SKIPPED
            The phone number is not registered.

        CONTACT_SUCCESS
            The Add Contact popup has disappeared without
            displaying the unregistered message.

    Raises:
        TimeoutException
            Neither result appeared within the timeout.

    Important:
        A timeout is NOT converted into "skipped".
        It is allowed to become a failed operation so that
        Section 10 can retry it.
    """

    def detect_result(d):
        # --------------------------------------------
        # Result 1: unregistered phone number
        # --------------------------------------------
        if unregistered_phone_message_is_visible(d):
            return CONTACT_SKIPPED

        # --------------------------------------------
        # Result 2: Add Contact popup disappeared
        # --------------------------------------------
        #
        # The tested code already uses this exact popup
        # locator and waits for it to become invisible.
        #
        popup_elements = d.find_elements(
            *ADD_CONTACT_POPUP
        )

        if not popup_elements:
            return CONTACT_SUCCESS

        try:
            popup_visible = any(
                element.is_displayed()
                for element in popup_elements
            )
        except WebDriverException:
            popup_visible = False

        if not popup_visible:
            return CONTACT_SUCCESS

        return False

    return wait_until(
        driver,
        detect_result,
        timeout,
    )


# ------------------------------------------------------------
# Helper: close the unregistered-phone message
# ------------------------------------------------------------

def close_unregistered_phone_popup(driver):
    """
    Close the Eitaa 'phone number is not registered' popup.

    The project specification requires the popup's × button
    to be clicked before closing the Add Contact window.

    Because Section 6 does not currently provide a tested
    selector for this specific × button, the locator below
    intentionally searches for a close button inside the
    visible element containing the unregistered message.

    If the actual Eitaa DOM uses a different structure, only
    this helper needs to be adjusted.
    """

    message_element = wait_visible(
        driver,
        (
            By.XPATH,
            "//*[contains(normalize-space(.), "
            f"{UNREGISTERED_PHONE_TEXT!r}"
            ")]",
        ),
    )

    # Find the nearest popup/dialog-like ancestor and then
    # locate its close button.
    #
    # This is the only DOM relationship in Section 8 that
    # was not supplied as a tested Section 6 selector.

    close_button = wait_clickable(
        driver,
        (
            By.XPATH,
            "("
            "ancestor-or-self::*["
            "contains(@class,'popup') "
            "or contains(@class,'modal') "
            "or contains(@class,'dialog')"
            "][1]"
            ")"
            "//button["
            "contains(@class,'popup-close') "
            "or contains(@class,'close') "
            "or contains(@class,'btn-icon')"
            "]"
            "[1]",
        ),
    )

    close_button.click()

    # Do not proceed until the message is actually gone.
    wait_until(
        driver,
        lambda d: not unregistered_phone_message_is_visible(d),
    )


# ------------------------------------------------------------
# Helper: close Add Contact window
# ------------------------------------------------------------

def close_add_contact_window(driver):
    """
    Close the Add Contact window using the tested Section 6
    selector.
    """

    close_button = wait_clickable(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
    )

    close_button.click()

    # Wait until the Add Contact window is no longer active.
    wait_invisible(
        driver,
        ADD_CONTACT_POPUP,
    )


# ------------------------------------------------------------
# Main Section 8 function
# ------------------------------------------------------------

def add_contact(driver, full_name, phone_normalized):
    """
    Add one contact to Eitaa.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    full_name : str
        Contact's name.

    phone_normalized : str
        Normalized Iranian mobile number.

    Returns
    -------
    str
        One of:

            CONTACT_SUCCESS
            CONTACT_SKIPPED
            CONTACT_FAILED

    Notes
    -----
    This function does not modify the database.

    Section 10 will decide whether a failed result should be
    retried.
    """

    try:

        # ====================================================
        # 1. Make sure we are in Contacts
        # ====================================================

        ensure_contacts_page(driver)


        # ====================================================
        # 2. Open Add Contact
        # ====================================================

        wait_and_click(
            driver,
            Selectors.ADD_CONTACT_BUTTON,
        )


        # ====================================================
        # 3. Wait for the Add Contact form
        # ====================================================

        wait_visible(
            driver,
            Selectors.FIRST_NAME,
        )

        wait_visible(
            driver,
            Selectors.PHONE,
        )


        # ====================================================
        # 4. Enter contact name
        # ====================================================

        wait_and_type(
            driver,
            Selectors.FIRST_NAME,
            full_name,
        )


        # ====================================================
        # 5. Enter normalized phone number
        # ====================================================

        wait_and_type(
            driver,
            Selectors.PHONE,
            phone_normalized,
        )


        # ====================================================
        # 6. Click Save
        # ====================================================

        wait_and_click(
            driver,
            Selectors.SAVE_BUTTON,
        )


        # ====================================================
        # 7. Determine Eitaa result
        # ====================================================

        result = wait_for_add_contact_result(driver)


        # ====================================================
        # 8. Unregistered phone
        # ====================================================

        if result == CONTACT_SKIPPED:

            close_unregistered_phone_popup(driver)

            close_add_contact_window(driver)

            return CONTACT_SKIPPED


        # ====================================================
        # 9. Successfully added
        # ====================================================

        if result == CONTACT_SUCCESS:

            close_add_contact_window(driver)

            return CONTACT_SUCCESS


        # ====================================================
        # 10. Defensive fallback
        # ====================================================

        return CONTACT_FAILED


    except Exception as exc:

        # ----------------------------------------------------
        # Best-effort cleanup
        # ----------------------------------------------------
        #
        # Do not allow cleanup errors to hide the original
        # failure.
        #
        # Section 10 will receive "failed" and can retry the
        # contact.

        try:
            close_add_contact_window(driver)
        except Exception:
            pass

        print(
            f"Add contact failed "
            f"for {full_name!r} / {phone_normalized!r}: "
            f"{type(exc).__name__}: {exc}"
        )

        return CONTACT_FAILED


In [33]:
# ============================================================
# 9- Fetching the Next Contact
# ============================================================
def get_next_contact(conn):
    cursor = conn.cursor(dictionary=True)

    try:
        cursor.execute("""
            SELECT
                id,
                full_name,
                phone_original,
                phone_normalized,
                attempts
            FROM contacts
            WHERE status IN ('pending', 'failed')
              AND attempts < 3
            ORDER BY id
            LIMIT 1
        """)

        return cursor.fetchone()

    finally:
        cursor.close()

# You may want failed contacts handled separately rather than automatically retried. For 5,000 records, I generally recommend:

# pending → automatic retry
# failed  → max 2-3 attempts

# and then review the failures manually.

In [ ]:
# ============================================================
# 10- Complete processing loop
# ============================================================

import logging

from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException,
)

logging.basicConfig(
    filename="eitaa_import.log",
    level=logging.INFO,
    format=(
        "%(asctime)s | %(levelname)s | "
        "%(message)s"
    ),
)


def mark_processing(conn, contact_id):
    cursor = conn.cursor()

    try:
        cursor.execute("""
            UPDATE contacts
            SET
                status = 'processing',
                attempts = attempts + 1,
                started_at = NOW()
            WHERE id = %s
        """, (contact_id,))

        conn.commit()

    finally:
        cursor.close()


def mark_success(conn, contact_id):
    cursor = conn.cursor()

    try:
        cursor.execute("""
            UPDATE contacts
            SET
                status = 'success',
                error_message = NULL,
                completed_at = NOW()
            WHERE id = %s
        """, (contact_id,))

        conn.commit()

    finally:
        cursor.close()


def mark_failed(conn, contact_id, error):
    cursor = conn.cursor()

    try:
        cursor.execute("""
            UPDATE contacts
            SET
                status = 'failed',
                error_message = %s
            WHERE id = %s
        """, (
            str(error)[:60000],
            contact_id,
        ))

        conn.commit()

    finally:
        cursor.close()


def reset_stale_processing(conn):
    cursor = conn.cursor()

    try:
        cursor.execute("""
            UPDATE contacts
            SET status = 'pending'
            WHERE status = 'processing'
              AND started_at <
                  NOW() - INTERVAL 30 MINUTE
        """)

        conn.commit()

    finally:
        cursor.close()


def run_import():
    conn = get_db_connection()

    # Use the Selenium driver created in Section 5.
    # Do not create another driver here.
    selenium_driver = globals().get("driver")

    if selenium_driver is None:
        raise RuntimeError(
            "Selenium driver from Section 5 is not available."
        )

    try:
        reset_stale_processing(conn)

        selenium_driver.get(
            "https://web.eitaa.com/"
        )

        # At this point, verify that the existing
        # Eitaa session is still authenticated.
        #
        # If not, stop and ask for manual login
        # rather than attempting OTP automatically.

        while True:

            contact = get_next_contact(conn)

            if contact is None:
                logging.info("No more contacts.")
                break

            contact_id = contact["id"]
            full_name = contact["full_name"]
            phone = contact["phone_normalized"]

            logging.info(
                "Processing id=%s name=%r phone=%s",
                contact_id,
                full_name,
                phone,
            )

            mark_processing(
                conn,
                contact_id,
            )

            try:

                add_contact(
                    selenium_driver,
                    full_name,
                    phone,
                )

                mark_success(
                    conn,
                    contact_id,
                )

                logging.info(
                    "SUCCESS id=%s",
                    contact_id,
                )

            except TimeoutException as exc:

                mark_failed(
                    conn,
                    contact_id,
                    f"Timeout: {exc}",
                )

                logging.exception(
                    "TIMEOUT id=%s",
                    contact_id,
                )

            except WebDriverException as exc:

                mark_failed(
                    conn,
                    contact_id,
                    f"WebDriver error: {exc}",
                )

                logging.exception(
                    "WEBDRIVER ERROR id=%s",
                    contact_id,
                )

                # The current Selenium session is unusable.
                # Section 5 owns the driver, so do not attempt
                # to create another driver here.
                raise

            except Exception as exc:

                mark_failed(
                    conn,
                    contact_id,
                    repr(exc),
                )

                logging.exception(
                    "ERROR id=%s",
                    contact_id,
                )

    finally:

        # Section 11 does NOT quit the Selenium driver.
        # Section 6 is responsible for closing the driver
        # and removing the temporary Chrome profile.

        conn.close()


run_import()


In [25]:
# Close Add Contact window
close_contact_window = wait_clickable(
    driver,
    Selectors.CLOSE_CONTACT_WINDOW
)

close_contact_window.click()

In [ ]:
# ============================================================
# Chrome crash recovery
# ============================================================
except WebDriverException:

logging.exception(
    "Chrome/WebDriver failure"
)

try:
    driver.quit()
except Exception:
    pass

driver = create_driver()

driver.get(
    "https://web.eitaa.com/"
)

# Wait for authenticated Eitaa state again.

In [ ]:
# ============================================================
# Add screenshots for failures
# ============================================================
from pathlib import Path
from datetime import datetime

SCREENSHOT_DIR = Path("screenshots")
SCREENSHOT_DIR.mkdir(exist_ok=True)


def save_failure_screenshot(driver, contact_id):
    filename = (
            SCREENSHOT_DIR
            / f"{contact_id}_{datetime.now():%Y%m%d_%H%M%S}.png"
    )

    driver.save_screenshot(str(filename))

    return filename




In [ ]:
except Exception as exc:

screenshot = save_failure_screenshot(
    driver,
    contact_id,
)

mark_failed(
    conn,
    contact_id,
    f"{exc}\nScreenshot: {screenshot}",
)